# GameSense Kaggle Steam Store content bundle

This notebook prepares the first **local** GameSense model-and-catalog bundle from Kaggle **Steam Store Games (Clean dataset)**: <https://www.kaggle.com/datasets/nikdavis/steam-store-games>.

Scope and disclosures:

- Kaggle lists the dataset as **Attribution 4.0 International (CC BY 4.0)**; the bundle and app disclosure preserve attribution.
- The Kaggle page describes the source as Steam Store/SteamSpy-derived and gathered around May 2019. Upstream Steam/SteamSpy terms uncertainty remains unresolved and was accepted only for this local notebook/app-demo scope.
- This notebook fits sparse TF-IDF vocabulary/IDF/item vectors and uses source tags/genres. It does **not** fine-tune a neural model, learn from users, claim personalization, or claim current/live Steam trends.
- Keep raw Kaggle files and generated bundle files out of git. Download the zip bundle and run report from `/kaggle/working` and return them for local import.


In [ ]:
from pathlib import Path
import json
import os
import sys

# If the notebook is inside the GameSense repo, this default works. If you upload the repo
# elsewhere in Kaggle, set GAMESENSE_REPO_ROOT before running.
REPO_ROOT = Path(os.getenv('GAMESENSE_REPO_ROOT', '.')).resolve()
assert (REPO_ROOT / 'backend/app/ml_pipeline.py').exists(), f'GameSense repo not found at {REPO_ROOT}'
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'backend'))
print('repo:', REPO_ROOT)


In [ ]:
from app.ml_pipeline import steam_store_source_check

MANIFEST_PATH = Path('data/source-manifests/kaggle.steam-store-games-clean.json')
source_report = steam_store_source_check(MANIFEST_PATH)
print(json.dumps(source_report, indent=2, sort_keys=True))
assert source_report['ok'], 'Source scope check failed; stop rather than bypassing source validation.'


In [ ]:
configured = os.getenv('GAMESENSE_STEAM_STORE_DATASET_DIR')
if configured:
    DATASET_DIR = Path(configured)
else:
    DATASET_DIR = Path('/kaggle/input/steam-store-games')

expected_files = ['steam.csv', 'steam_description_data.csv', 'steamspy_tag_data.csv', 'steam_media_data.csv']
print('dataset dir:', DATASET_DIR)
for name in expected_files:
    path = DATASET_DIR / name
    print(name, 'exists=' + str(path.exists()), 'path=' + str(path))
assert (DATASET_DIR / 'steam.csv').exists(), 'Attach Kaggle Steam Store Games dataset or set GAMESENSE_STEAM_STORE_DATASET_DIR.'


In [ ]:
import hashlib
from app.ml_pipeline import normalize_steam_store_games, validate_normalized, write_json

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else (REPO_ROOT / 'artifacts/models/local-kaggle-run')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_hashes = {name: sha256_file(DATASET_DIR / name) for name in expected_files if (DATASET_DIR / name).exists()}
normalized = normalize_steam_store_games(MANIFEST_PATH, DATASET_DIR)
normalized_path = OUTPUT_DIR / 'gamesense-steam-normalized-catalog.json'
write_json(normalized_path, normalized)
validation = validate_normalized(MANIFEST_PATH, normalized_path)
validation_path = OUTPUT_DIR / 'gamesense-steam-validation-report.json'
write_json(validation_path, validation)
print(json.dumps({
    'rowCount': normalized['rowCount'],
    'rawRowCount': normalized['rawRowCount'],
    'skippedRows': normalized['skippedRows'],
    'validationOk': validation['ok'],
    'invalidRows': validation['invalidRowCount'],
    'sourceHashes': source_hashes,
}, indent=2, sort_keys=True))
assert validation['ok'], 'Normalized catalog failed validation; inspect validation report before building a bundle.'


In [ ]:
from collections import Counter

all_genres = [genre for game in normalized['games'] for genre in game['genres']]
all_tags = [tag for game in normalized['games'] for tag in game['tags']]
missing_descriptions = sum(1 for game in normalized['games'] if not game.get('description'))
evaluation_summary = {
    'rowCount': normalized['rowCount'],
    'missingDescriptions': missing_descriptions,
    'gamesWithGenres': sum(1 for game in normalized['games'] if game['genres']),
    'gamesWithTags': sum(1 for game in normalized['games'] if game['tags']),
    'topGenres': Counter(all_genres).most_common(15),
    'topTags': Counter(all_tags).most_common(15),
    'validationOk': validation['ok'],
    'note': 'Sanity summary only; no tuned accuracy or real-user relevance evaluation is claimed.',
}
print(json.dumps(evaluation_summary, indent=2, sort_keys=True))


In [ ]:
from app.ml_bundle import build_tfidf_content_bundle, load_content_bundle

source_manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
bundle_path = OUTPUT_DIR / 'gamesense-steam-store-content-bundle.zip'
bundle_manifest = build_tfidf_content_bundle(
    games=normalized['games'],
    source_manifest=source_manifest,
    output_path=bundle_path,
    source_file_hashes=source_hashes,
    evaluation_summary=evaluation_summary,
)
loaded = load_content_bundle(bundle_path)
print(json.dumps({
    'bundle': str(bundle_path),
    'bundleSha256': bundle_manifest['bundleSha256'],
    'catalogItems': len(loaded.catalog),
    'vocabularySize': len(loaded.vocabulary),
    'dataSnapshot': loaded.data_snapshot,
    'modelVersion': loaded.model_version,
}, indent=2, sort_keys=True))


In [ ]:
from app.recommender import recommend_prefit_content

# Basic sanity check: if a seed is used, it must not recommend itself and all IDs must
# resolve against the same loaded catalog snapshot.
seed = loaded.catalog[0]
tokens_by_index = [token for token, _ in sorted(loaded.vocabulary.items(), key=lambda item: item[1])]
recommendations = recommend_prefit_content(
    loaded.catalog,
    mode='similar',
    seed_slug=seed.slug,
    item_vectors=loaded.item_vectors,
    vocabulary=loaded.vocabulary,
    idf=loaded.idf,
    tokens_by_index=tokens_by_index,
    catalog_scores=loaded.catalog_scores(),
    limit=5,
)
print('seed:', seed.slug, seed.title)
for item in recommendations:
    print(item.slug, item.title, item.score, item.componentRanks)
assert seed.slug not in [item.slug for item in recommendations]
assert all(item.slug in {game.slug for game in loaded.catalog} for item in recommendations)


In [ ]:
run_report = {
    'sourceReport': source_report,
    'datasetDir': str(DATASET_DIR),
    'sourceFileHashes': source_hashes,
    'normalizedCatalogPath': str(normalized_path),
    'validationReportPath': str(validation_path),
    'bundlePath': str(bundle_path),
    'bundleSha256': bundle_manifest['bundleSha256'],
    'bundleManifest': bundle_manifest,
    'evaluationSummary': evaluation_summary,
    'nextLocalImport': {
        'copyBundleOutsideGit': 'Place gamesense-steam-store-content-bundle.zip under a local ignored path such as artifacts/models/local-kaggle-run/.',
        'backendEnv': 'RECOMMENDER_MODE=content CONTENT_BUNDLE_PATH=/absolute/path/to/gamesense-steam-store-content-bundle.zip',
        'verify': 'Open the UI, select a catalog game card, and inspect its Similar recommendations. The API also supports Cozy and Competitive modes.',
    },
}
run_report_path = OUTPUT_DIR / 'gamesense-steam-store-kaggle-run-report.json'
run_report_path.write_text(json.dumps(run_report, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('Download these two files and return them:')
print('-', bundle_path)
print('-', run_report_path)
